# 08mareyfeeds: Madrid to Barcelona, one day, three operators (GTFS)

**What it does:** gets the timetables of the three high-speed operators between Madrid and Barcelona and keeps every train that runs between the two cities on one normal weekday, **Wednesday 25 November 2026**, with its times at every station it calls at. This is the input for the Marey diagram (part 2): one line per train, time across, stations down.

- **Renfe:** today's official feed, already downloaded by `00renfegtfs` (`data/raw/renfe/2026-11-16/`).
- **Ouigo:** official feed (Spanish National Access Point, via the Mobility Database archive, feed mdb-2785, version of 3 Jul 2026, service 26 Jun to 13 Dec 2026).
- **Iryo:** Iryo publishes no open feed (not on the National Access Point, checked 20 Nov). Community feed built from Iryo's own data by Clement Desouche (deryclem.github.io/iryo-gtfs), Mobility Database mdb-3645, version of 19 Nov 2026 (service 5 Nov 2026 to 11 Dec 2027). **Checked against ticket listings (Kombo, read 20 Nov):** Wed 2 Dec, all 14 + 14 Iryo trains match the feed to the minute; Wed 25 Nov, every Iryo train on sale matches the feed; 6 feed trains were not on sale that day (mostly 3 h 02 nonstops, likely sold out), and the same trains are on sale a week later.
- A train counts if it calls at a Madrid station (Atocha or Chamartin) and at Barcelona Sants. Only the part between the two is kept.

**Inputs:** `data/raw/renfe/2026-11-16/` (from `00renfegtfs`), the internet (files.mobilitydatabase.org) for Ouigo and Iryo.
**Outputs:** `data/raw/ouigo/<version>/`, `data/raw/iryo/<version>/` (zip and unpacked files), `data/clean/marey_trips.csv` (one row per train and station).

**Data credit:** Renfe Viajeros open data (data.renfe.com). Ouigo Espana via the Punto de Acceso Nacional (nap.transportes.gob.es, licence nap.transportes.gob.es/licencia-datos). Iryo: community feed by Clement Desouche, "via data from iryo" (licence printed in section 2). Archives: Mobility Database (MobilityData).

## 0. Settings

In [1]:
from pathlib import Path
import datetime
import unicodedata
import zipfile
import requests
import pandas as pd

RAW_DIR = Path("data") / "raw"
CLEAN_DIR = Path("data") / "clean"
DAY = "2026-11-25"  # <- the day shown (a normal Wednesday; same day as the 2026 era)
FEEDS = {
    "Renfe": {"folder": RAW_DIR / "renfe" / "2026-11-16", "url": None},  # <- already downloaded by 00renfegtfs
    "Ouigo": {"folder": RAW_DIR / "ouigo" / "2026-07-03", "url": "https://files.mobilitydatabase.org/mdb-2785/mdb-2785-202607030044/mdb-2785-202607030044.zip"},  # <- Mobility Database mdb-2785
    "Iryo": {"folder": RAW_DIR / "iryo" / "2026-11-19", "url": "https://files.mobilitydatabase.org/mdb-3645/mdb-3645-202611192230/mdb-3645-202611192230.zip"},  # <- Mobility Database mdb-3645 (community feed)
}
MADRID_WORDS = ["puerta de atocha", "madrid atocha", "chamartin"]  # <- a stop is a Madrid station if its name contains one of these (lower case, no accents)
BARCELONA_WORDS = ["barcelona sants", "barcelona-sants"]  # <- the same for Barcelona Sants
STATIONS = [  # <- corridor stations from Madrid to Barcelona: (short name, words in the stop name)
    ("Madrid Chamartin", ["chamartin"]),
    ("Madrid Atocha", ["puerta de atocha", "madrid atocha"]),  # <- Renfe and Ouigo say "Puerta de Atocha", Iryo "Madrid Atocha"
    ("Guadalajara", ["guadalajara"]),
    ("Calatayud", ["calatayud"]),
    ("Zaragoza", ["zaragoza"]),
    ("Lleida", ["lleida"]),
    ("Tarragona", ["camp de tarragona"]),
    ("Barcelona Sants", ["barcelona sants", "barcelona-sants"]),
]
# Ticket listings read on Kombo (kombo.co) on 20 Nov 2026, Madrid departures (section 5 compares)
LISTED = {
    ("Iryo", "2026-11-25"): ["06:09", "07:22", "08:22", "09:12", "09:57", "11:22", "12:22", "14:42", "17:22", "18:32", "19:29"],  # <- trains on sale on 20 Nov
    ("Iryo", "2026-12-02"): ["06:09", "07:22", "08:22", "08:50", "09:12", "09:57", "11:22", "12:22", "14:42", "15:52", "16:37", "17:22", "18:32", "19:29"],  # <- same weekday, a week later
    ("Ouigo", "2026-11-25"): ["06:22", "09:31", "13:42", "16:22", "20:10"],
    ("Renfe", "2026-11-25"): ["06:16", "06:27", "06:57", "07:27", "07:57", "08:27", "08:57", "09:27", "10:27", "11:27", "12:27", "12:50", "13:57", "14:27", "14:57", "15:27", "15:57", "16:27", "16:57", "17:27", "17:57", "18:27", "18:57", "19:34", "20:30"],
}

## 1. Download (skipped for feeds already there)

In [2]:
for operator, feed in FEEDS.items():
    folder = feed["folder"]
    zip_path = folder / "google_transit.zip"
    if zip_path.exists():
        print(operator, "already downloaded:", zip_path)
        continue
    if feed["url"] is None:
        raise FileNotFoundError(f"{zip_path} is missing: run 00renfegtfs first.")
    folder.mkdir(parents=True, exist_ok=True)
    response = requests.get(feed["url"], headers={"User-Agent": "Mozilla/5.0"}, timeout=180)
    response.raise_for_status()
    zip_path.write_bytes(response.content)
    print(operator, "downloaded", round(len(response.content) / 1e6, 2), "MB to", zip_path)

Renfe already downloaded: data\raw\renfe\2026-11-16\google_transit.zip
Ouigo already downloaded: data\raw\ouigo\2026-07-03\google_transit.zip
Iryo already downloaded: data\raw\iryo\2026-11-19\google_transit.zip


## 2. Unpack, files, dates covered and licences

In [3]:
def read_gtfs(folder, name):
    table = pd.read_csv(Path(folder) / name, dtype=str, keep_default_na=False, encoding="utf-8-sig", skipinitialspace=True)
    table.columns = [c.strip() for c in table.columns]
    for column in table.columns:
        table[column] = table[column].str.strip()
    return table


def plain(text):
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.lower()


files = {}
for operator, feed in FEEDS.items():
    folder = feed["folder"]
    with zipfile.ZipFile(folder / "google_transit.zip") as z:
        z.extractall(folder)
        files[operator] = z.namelist()
    print()
    print(operator, "-", folder)
    for name in files[operator]:
        if name.endswith(".txt") and name not in ("LICENSE.txt", "licences_and_sources.txt"):
            print(f"  {name:24s} {len(read_gtfs(folder, name)):8d} rows")
        else:
            print(f"  {name}")
    if "calendar.txt" in files[operator]:
        calendar = read_gtfs(folder, "calendar.txt")
        print("  calendar: services from", calendar["start_date"].min(), "to", calendar["end_date"].max())
    print("  operators:", list(read_gtfs(folder, "agency.txt")["agency_name"]))
    if "LICENSE.txt" in files[operator]:
        print("  --- LICENSE.txt ---")
        print((folder / "LICENSE.txt").read_text(encoding="utf-8", errors="replace"))
    if "licences_and_sources.txt" in files[operator]:
        text = (folder / "licences_and_sources.txt").read_text(encoding="utf-8", errors="replace")
        lines = [l for l in text.splitlines() if operator.lower() in l.lower()]
        print(f"  --- licences_and_sources.txt: a general file of the export tool (other operators' sources); lines naming {operator}: {lines if lines else 'none'}")
        print("      (Ouigo's own terms: the National Access Point licence, nap.transportes.gob.es/licencia-datos)")


Renfe - data\raw\renfe\2026-11-16
  agency.txt                      1 rows
  calendar_dates.txt          24619 rows
  calendar.txt                 6402 rows
  routes.txt                    671 rows
  stops.txt                    1006 rows
  stop_times.txt              51846 rows
  trips.txt                    6402 rows
  calendar: services from 20261115 to 20270929
  operators: ['RENFE OPERADORA']

Ouigo - data\raw\ouigo\2026-07-03
  trips.txt                      69 rows
  routes.txt                     11 rows
  calendar.txt                   27 rows
  calendar_dates.txt             67 rows
  transfers.txt                  16 rows
  licences_and_sources.txt
  stop_times.txt                233 rows
  country_names.txt              38 rows
  agency.txt                      1 rows
  stops.txt                      16 rows
  calendar: services from 20260626 to 20261212
  operators: ['OUIGO Espana']
  --- licences_and_sources.txt: a general file of the export tool (other operators' source

## 3. Services running on the day

`calendar.txt` gives weekly patterns with start and end dates; `calendar_dates.txt` adds (type 1) or removes (type 2) single dates.

In [4]:
day = datetime.date.fromisoformat(DAY)
day_code = day.strftime("%Y%m%d")
weekday = day.strftime("%A").lower()


def services_on(folder, names):
    running = set()
    if "calendar.txt" in names:
        calendar = read_gtfs(folder, "calendar.txt")
        on = (calendar["start_date"] <= day_code) & (calendar["end_date"] >= day_code) & (calendar[weekday] == "1")
        running = set(calendar.loc[on, "service_id"])
    if "calendar_dates.txt" in names:
        dates = read_gtfs(folder, "calendar_dates.txt")
        dates = dates[dates["date"] == day_code]
        running |= set(dates.loc[dates["exception_type"] == "1", "service_id"])
        running -= set(dates.loc[dates["exception_type"] == "2", "service_id"])
    return running


running = {}
for operator, feed in FEEDS.items():
    running[operator] = services_on(feed["folder"], files[operator])
    print(f"{operator}: {len(running[operator])} services run on {weekday.title()} {DAY}")

Renfe: 1610 services run on Wednesday 2026-11-25
Ouigo: 9 services run on Wednesday 2026-11-25
Iryo: 22 services run on Wednesday 2026-11-25


## 4. Trains between Madrid and Barcelona that day

For each trip running that day: keep it if it calls at a Madrid station and at Barcelona Sants, and keep only its stops from one to the other. Times are minutes after midnight (GTFS times can pass 24:00).

In [5]:
def minutes(text):
    if not text:
        return float("nan")
    h, m, s = (int(x) for x in text.split(":"))
    return h * 60 + m + s / 60


def station_of(name):
    p = plain(name)
    for short, words in STATIONS:
        if any(w in p for w in words):
            return short
    return ""


rows = []
for operator, feed in FEEDS.items():
    folder = feed["folder"]
    stops = read_gtfs(folder, "stops.txt")
    stop_name = dict(zip(stops["stop_id"], stops["stop_name"]))
    madrid = {s for s, n in stop_name.items() if any(w in plain(n) for w in MADRID_WORDS)}
    barcelona = {s for s, n in stop_name.items() if any(w in plain(n) for w in BARCELONA_WORDS)}
    print(f"{operator}: Madrid stops {sorted(stop_name[s] for s in madrid)}; Barcelona stops {sorted(stop_name[s] for s in barcelona)}")
    trips = read_gtfs(folder, "trips.txt")
    trips = trips[trips["service_id"].isin(running[operator])]
    times = read_gtfs(folder, "stop_times.txt")
    times = times[times["trip_id"].isin(set(trips["trip_id"]))].copy()
    times["seq"] = times["stop_sequence"].astype(int)
    times = times.sort_values(["trip_id", "seq"])
    number = dict(zip(trips["trip_id"], trips["trip_short_name"] if "trip_short_name" in trips.columns else trips["trip_id"]))
    for trip_id, calls in times.groupby("trip_id", sort=False):
        ids = list(calls["stop_id"])
        at_m = [i for i, s in enumerate(ids) if s in madrid]
        at_b = [i for i, s in enumerate(ids) if s in barcelona]
        if not at_m or not at_b:
            continue
        if at_m[0] < at_b[0]:
            first, last, direction = at_m[0], at_b[0], "to Barcelona"
        else:
            first, last, direction = at_b[0], at_m[-1], "to Madrid"
        part = calls.iloc[first:last + 1]
        train = number.get(trip_id) or trip_id
        for k, (_, c) in enumerate(part.iterrows()):
            rows.append({
                "operator": operator, "train": train, "trip_id": trip_id, "direction": direction, "seq": k,
                "stop_id": c["stop_id"], "stop_name": stop_name.get(c["stop_id"], ""), "station": station_of(stop_name.get(c["stop_id"], "")),
                "arr_min": minutes(c["arrival_time"]), "dep_min": minutes(c["departure_time"]),
            })
calls = pd.DataFrame(rows)

# One line per train: Renfe sometimes lists a train twice under the same number and times (e.g. one part runs on beyond Barcelona Sants). Keep the first.
first_call = calls[calls["seq"] == 0].set_index("trip_id")
key = first_call[["operator", "direction", "train", "dep_min"]].apply(tuple, axis=1)
keep = set(key.drop_duplicates().index)
print(f"same train listed twice, dropped: {key.size - len(keep)}")
calls = calls[calls["trip_id"].isin(keep)].reset_index(drop=True)
unknown = sorted(set(calls.loc[calls["station"] == "", "stop_name"]))
print(f"stops not in STATIONS (kept, station left blank): {unknown if unknown else 'none'}")

Renfe: Madrid stops ['Madrid-Chamartín-Clara Campoamor', 'Madrid-Puerta de Atocha-Almudena Grandes']; Barcelona stops ['Barcelona-Sants']
Ouigo: Madrid stops ['MADRID-CHAMARTIN', 'MADRID-PUERTA DE ATOCHA']; Barcelona stops ['BARCELONA-SANTS']
Iryo: Madrid stops ['Madrid Atocha', 'Madrid Chamartín']; Barcelona stops ['Barcelona Sants']
same train listed twice, dropped: 12
stops not in STATIONS (kept, station left blank): none


## 5. Summary and checks

In [6]:
def clock(m):
    m = int(round(m))
    return f"{m // 60:02d}:{m % 60:02d}"


trains = calls.groupby("trip_id").agg(operator=("operator", "first"), train=("train", "first"), direction=("direction", "first"),
                                      dep=("dep_min", "first"), arr=("arr_min", "last"), stops=("station", lambda s: " / ".join(s.iloc[1:-1])))
trains["minutes"] = trains["arr"] - trains["dep"]
trains = trains.sort_values(["direction", "dep"])
print(trains.groupby(["direction", "operator"]).agg(trains=("train", "size"), first=("dep", lambda x: clock(x.min())), last=("dep", lambda x: clock(x.max())),
                                                  fastest=("minutes", "min"), median=("minutes", "median")).to_string())
print(f"all trains: {len(trains)}")
for direction in ["to Barcelona", "to Madrid"]:
    print()
    print("---", direction)
    part = trains[trains["direction"] == direction]
    for _, t in part.iterrows():
        print(f"  {clock(t['dep'])} {clock(t['arr'])} {int(t['minutes']):4d} min  {t['operator']:6s} {t['train']:>7s}  {t['stops'] if t['stops'] else 'nonstop'}")

print()
print("Check against ticket listings (Madrid departures):")
for (operator, date), listed in LISTED.items():
    if date != DAY:
        print(f"  {operator} {date}: listing for another date, not compared (feed shown for {DAY})")
        continue
    feed_times = set(trains.loc[(trains["operator"] == operator) & (trains["direction"] == "to Barcelona"), "dep"].map(clock))
    print(f"  {operator} {date}: listed {len(listed)}, in feed {len(feed_times)}; listed but not in feed: {sorted(set(listed) - feed_times) or 'none'}; in feed but not listed: {sorted(feed_times - set(listed)) or 'none'}")

                       trains  first   last  fastest  median
direction    operator                                       
to Barcelona Iryo          14  06:09  19:29    182.0   197.0
             Ouigo          5  06:22  20:10    182.0   198.0
             Renfe         25  06:16  20:30    182.0   197.0
to Madrid    Iryo          14  05:45  19:40    182.0   197.0
             Ouigo          5  07:35  20:20    182.0   197.0
             Renfe         26  05:40  20:00    182.0   197.0
all trains: 89

--- to Barcelona
  06:09 09:35  206 min  Iryo     06261  Zaragoza / Tarragona
  06:16 09:24  188 min  Renfe    03301  Tarragona
  06:22 09:45  203 min  Ouigo  TGV 6471  Zaragoza
  06:27 09:50  203 min  Renfe    03063  Calatayud / Zaragoza
  06:57 09:59  182 min  Renfe    03071  nonstop
  07:22 10:24  182 min  Iryo     06071  nonstop
  07:27 11:11  224 min  Renfe    03073  Guadalajara / Zaragoza / Lleida / Tarragona
  07:57 10:59  182 min  Renfe    03081  nonstop
  08:22 11:24  182 min  Iryo 

## 6. Save

In [7]:
out = calls.copy()
out["arr_min"] = out["arr_min"].round(1)
out["dep_min"] = out["dep_min"].round(1)
out.to_csv(CLEAN_DIR / "marey_trips.csv", index=False)
print(f"saved marey_trips.csv: {len(out)} rows, {out['trip_id'].nunique()} trains")

saved marey_trips.csv: 300 rows, 89 trains


___
**END**